# Notion-Zotero Analysis

Full pipeline: load canonical bundles → build summary tables → clean → explore.

**Data sources** (pick one):
-  — bundles pulled via 
-  — bundles pulled via 
-  — bundles parsed from local fixture files

In [4]:
from pathlib import Path
import pandas as pd

from notion_zotero.analysis import (
    load_canonical_records,
    build_summary_dataframes,
    clean_table,
    run_analysis,
)

# Load from live pulled data
pulled_dir = Path("data/pulled/notion/learning_analytics_review")
if not pulled_dir.exists() or not any(pulled_dir.glob("*.canonical.json")):
    raise FileNotFoundError(
        f"No pulled data found in {pulled_dir}. "
        "Run: notion-zotero pull-notion"
    )

bundles = load_canonical_records(str(pulled_dir))
print(f"Loaded {len(bundles)} bundles from {pulled_dir}")

Loaded 442 bundles from data\pulled\notion\learning_analytics_review


In [5]:
# --- Cell 2: Inspect loaded bundles ---
print(f"Loaded {len(bundles)} canonical bundles")

if bundles:
    sample = bundles[0]
    refs = sample.get("references", [])
    if refs:
        r = refs[0]
        print(f"Sample: {r.get("title", "(no title)")} ({r.get("year", "?")})")

Loaded 442 canonical bundles
Sample: Contrastive Personalized Exercise Recommendation With Reinforcement Learning (2024)


In [ ]:
# --- Cell 3: Build summary tables (accepted papers only) ---
def my_label_fn(task_name):
    mapping = {
        "performance_prediction": "PRED",
        "descriptive_modelling": "DESC",
        "recommender_systems": "REC",
        "knowledge_tracing": "KT",
    }
    return mapping.get(task_name.lower(), task_name)

def _is_accepted(bundle):
    """Return True if the bundle has an accepted status (or no status info)."""
    ws = bundle.get("workflow_states") or []
    if not ws:
        # fall back to sync_metadata Status
        refs = bundle.get("references") or [{}]
        sm = (refs[0].get("sync_metadata") or {})
        status = sm.get("notion_properties", {}).get("Status") or ""
    else:
        status = (ws[0].get("state") or "")
    if not status:
        return True  # no status info — include by default
    return "accepted" in status.lower()

accepted_bundles = [b for b in bundles if _is_accepted(b)]
print(f"Accepted: {len(accepted_bundles)} / {len(bundles)} total bundles")

dfs = build_summary_dataframes(accepted_bundles, task_label_fn=my_label_fn)
for name, df in dfs.items():
    print(f"  {name}: {len(df)} rows x {len(df.columns)} cols")
dfs.get("Reading List")

In [7]:
# --- Cell 4: Clean tables (caller-provided fix dicts) ---
MY_TYPO_FIXES = {r"Fiiltering": "Filtering", r"Exerrcise": "Exercise"}
MY_VALUE_MAP  = {"n/a": "Not Applicable", "none": "Not Applicable", "na": "Not Applicable"}
SEARCH_COLS   = ["Search Strategy"]

cleaned_dfs = {}
for name, df in dfs.items():
    cleaned, log = clean_table(df, typo_fixes=MY_TYPO_FIXES, value_map=MY_VALUE_MAP, search_strategy_columns=SEARCH_COLS)
    cleaned_dfs[name] = cleaned
    print(f"  {name}: {log["text_updates"]} cell(s) normalised")

  Reading List: 0 cell(s) normalised


In [8]:
# --- Cell 5: Shortcut — full pipeline ---
raw_dfs, clean_dfs, norm_log = run_analysis(
    pulled_dir,
    task_label_fn=my_label_fn,
    typo_fixes=MY_TYPO_FIXES,
    value_map=MY_VALUE_MAP,
    search_strategy_columns=SEARCH_COLS,
)
pd.DataFrame(norm_log).T

,rows_before,rows_after,text_updates,search_strategy_updates
Reading List,442,442,0,0


In [10]:
# --- Cell 6: Explore ---
rl = clean_dfs.get("Reading List", pd.DataFrame())
if not rl.empty and "year" in rl.columns:
    display(rl["year"].value_counts().sort_index())
if not rl.empty and "journal" in rl.columns:
    display(rl["journal"].dropna().value_counts().head(10))
for name, df in clean_dfs.items():
    if name != "Reading List" and not df.empty:
        print(f"{name} ({len(df)} rows):")
        display(df.head(5))

year
1995     1
2004     1
2008     1
2009     2
2010     3
2012     5
2013     5
2014     9
2015    15
2016    12
2017    29
2018    29
2019    25
2020    49
2021    57
2022    49
2023    75
2024    59
2025    14
2026     2
Name: count, dtype: int64

journal
LAK                                           54
IEEE Transactions on Learning Technologies    27
Computers & Education                         24
IEEE Access                                   15
Education and Information Technologies        14
Computers in Human Behavior                   14
Knowledge-Based Systems                       13
Applied Sciences                              11
ArXiv                                         11
Internet and Higher Education                 10
Name: count, dtype: int64